# HeartLens AI — Colab training & experiments

Runtime: **T4 GPU** (Edit → Notebook settings → Hardware accelerator → GPU).

Run the cells **top to bottom**. Each experiment is its own cell, so one
failure no longer kills the rest — fix, re-run that single cell, continue.
Results land in `heart-lens-training/results/`; models in `heart-lens-training/models/`.


In [ ]:
# 1. Get the code (fresh, non-nesting clone — safe to re-run)
%cd /content
!rm -rf heartlens
!git clone --depth 1 https://github.com/touhidsiddiqueeraj-bit/heartlens.git
%cd /content/heartlens

In [ ]:
# 2. Dependencies (only the few not preinstalled in Colab)
!pip install -q wfdb fpdf2

# Optional: keep the session alive during the long training cells
from IPython.display import display, Javascript
display(Javascript('''
function ClickConnect(){ colab.gcloud.drive.notebooks._refClick(1); }
setInterval(ClickConnect, 60000);
'''))

In [ ]:
# 3. Sanity: data split composition (expect all 3 classes in train/val/test)
%cd /content/heartlens/heart-lens-training
!python3 diagnose_data.py --data-dir ./mitdb 2>/dev/null | grep -E "bincount|records=|SKIP|nan" 

In [ ]:
# 4. Train robust denoiser + 3-class classifier, int8 sizes, PDF report
%cd /content/heartlens
!python3 auto_train.py --epochs 30 --max-per-class 3000

In [ ]:
# Exp 1: grouped patient-level CV (5 folds x 3 seeds)
%cd /content/heartlens/heart-lens-training
!python3 group_kfold_eval.py --folds 5 --seeds 0,1,2 --epochs 30

In [ ]:
# Exp 2: noise robustness Raw/Filter/AE x SNR (also trains + saves robust_*.keras)
%cd /content/heartlens/heart-lens-training
!python3 evaluate_noise_robustness.py --epochs 30

In [ ]:
# Exp 3: external generalization mitdb -> SVDB + afdb
%cd /content/heartlens/heart-lens-training
!python3 external_validation.py --epochs 30

In [ ]:
# Exp 4: FP32 vs INT8 delta per architecture (CNN/LSTM/GRU/TCN) + size
%cd /content/heartlens/heart-lens-training
!python3 compare_models.py --epochs 30

In [ ]:
# Calibration: temperature scaling -> writes CALIB_TEMPERATURE to firmware Config.h
%cd /content/heartlens/heart-lens-training
!python3 calibrate.py --epochs 30 --write-config

In [ ]:
# Export int8 firmware models (robust_classifier_int8.tflite + robust_denoiser_int8.tflite)
%cd /content/heartlens/heart-lens-training
!python3 export_firmware_models.py

In [ ]:
# 5. Bundle results for download (models, results JSON, report PDF)
%cd /content/heartlens/heart-lens-training
!mkdir -p /content/out && cp -r models /content/out/ && cp -r results /content/out/
!cp ../auto_train_output/training_report.pdf /content/out/ 2>/dev/null; true
!cd /content/out && zip -qr heartlens_results.zip . && ls -lh heartlens_results.zip

## After downloading `heartlens_results.zip`

Unzip into the local repo (the assistant wires the firmware afterwards):

```
mkdir -p heart-lens-training
unzip ~/heartlens_results.zip -d heart-lens-training/   # or wherever you saved it
```

Expected healthy results after the cap fix:
- `group_kfold.json` — all three per-class F1s > 0, macro ~0.8
- `noise_robustness.json` — raw/filter/autoencoder macro F1 rising with SNR
- `external_validation.json` — SVDB report present (record 813 skipped)
- `model_comparison.json` — 4 rows, LSTM/GRU int8 converted (no CudnnRNNV3 error)
